# Model Evaluation

## Objective

The purpose of this notebook is to evaluate the performance of the Smart Recommendation & Personalization Engine.

Unlike traditional machine learning models, recommendation systems are evaluated using recommendation quality, ranking performance, and user interaction statistics instead of accuracy alone.

This notebook analyzes:

- Recommendation statistics
- Feedback distribution
- Personalization performance
- Ranking quality
- User-level recommendation behavior

The evaluation provides insights into how effectively the recommendation pipeline performs after incorporating simulated user feedback and personalization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display all columns while inspecting the dataset
pd.set_option("display.max_columns", None)

In [ ]:
recommendations_df = pd.read_csv(
    "../data/features/personalized_recommendations_final.csv"
)

recommendations_df.head()

In [ ]:
recommendations_df.info()

##### It consists of:

- Number of rows (recommendations)
- Number of columns (features).

In [ ]:
print("Dataset Shape:", recommendations_df.shape)

print("\nNumber of Rows:", recommendations_df.shape[0])

print("Number of Columns:", recommendations_df.shape[1])

In [ ]:
recommendations_df.columns

In [ ]:
missing_values = recommendations_df.isnull().sum()

missing_values

In [ ]:
duplicates = recommendations_df.duplicated().sum()

print("Duplicate Records:", duplicates)

In [ ]:
recommendations_df.describe()

In [ ]:
total_users = recommendations_df["visitorid"].nunique()

print(f"Total Users: {total_users}")

In [ ]:
total_products = recommendations_df["itemid"].nunique()

print(f"Unique Recommended Products: {total_products}")

In [ ]:
total_recommendations = len(recommendations_df)

print(f"Total Recommendation Records: {total_recommendations}")

In [ ]:
'''This metric calculates the average number of products recommended to each user.

It helps verify whether recommendations are distributed consistently across users.'''

avg_recommendations = (
    recommendations_df
    .groupby("visitorid")
    .size()
    .mean()
)

print(f"Average Recommendations per User: {avg_recommendations:.2f}")

In [ ]:
#The recommendation score is generated by the Recommendation Engine before feedback is incorporated.

avg_recommendation_score = (
    recommendations_df["recommendation_score"]
    .mean()
)

print(f"Average Recommendation Score: {avg_recommendation_score:.4f}")

In [ ]:
# The updated score is calculated after incorporating simulated user feedback.

avg_updated_score = (
    recommendations_df["updated_score"]
    .mean()
)

print(f"Average Updated Score: {avg_updated_score:.4f}")

In [ ]:
#The personalization score combines recommendation confidence with user preference information.

avg_personalization_score = (
    recommendations_df["personalization_score"]
    .mean()
)

print(f"Average Personalization Score: {avg_personalization_score:.4f}")

In [ ]:
'''To make interpretation easier, we summarize the key evaluation metrics in a single table.

This provides a quick overview of the recommendation system's performance.'''

evaluation_summary = pd.DataFrame({
    "Metric": [
        "Total Users",
        "Unique Products",
        "Recommendation Records",
        "Average Recommendations/User",
        "Average Recommendation Score",
        "Average Updated Score",
        "Average Personalization Score"
    ],
    "Value": [
        total_users,
        total_products,
        total_recommendations,
        round(avg_recommendations, 2),
        round(avg_recommendation_score, 4),
        round(avg_updated_score, 4),
        round(avg_personalization_score, 4)
    ]
})

evaluation_summary

In [ ]:
#Visualize Recommendation Scores

score_summary = pd.DataFrame({
    "Stage": [
        "Recommendation",
        "Feedback Updated",
        "Personalization"
    ],
    "Average Score": [
        avg_recommendation_score,
        avg_updated_score,
        avg_personalization_score
    ]
})

plt.figure(figsize=(8, 5))

plt.bar(
    score_summary["Stage"],
    score_summary["Average Score"]
)

plt.title("Average Scores Across Recommendation Pipeline")
plt.xlabel("Pipeline Stage")
plt.ylabel("Average Score")

plt.show()

In real-world recommendation systems, users interact with recommendations by:

- Clicking products
- Adding products to cart
- Purchasing products
- Rating products

Since the RetailRocket dataset does not contain future user feedback after recommendations are generated, feedback was simulated using recommendation confidence scores.

Evaluating the feedback distribution helps us understand how users respond to recommendations and how the system learns from those responses.

The first step is to count the number of Like, Neutral, and Dislike responses generated by the Feedback System.

In [ ]:
feedback_counts = (
    recommendations_df["feedback"]
    .value_counts()
)

feedback_counts

In [ ]:
feedback_percentage = (
    recommendations_df["feedback"]
    .value_counts(normalize=True)  #proportion (fraction)
    * 100
)

feedback_percentage.round(2)

In [ ]:
plt.figure(figsize=(8,5))

feedback_counts.plot(
    kind="bar"
)

plt.title("Feedback Distribution")

plt.xlabel("Feedback Type")

plt.ylabel("Count")

plt.xticks(rotation=0)

plt.show()

In [ ]:
plt.figure(figsize=(7,7))

feedback_percentage.plot(
    kind="pie",
    autopct="%1.1f%%"
)

plt.ylabel("")

plt.title("Feedback Percentage Distribution")

plt.show()

In [ ]:
recommendations_df["feedback_score"].describe()

In [ ]:
avg_feedback_score = (
    recommendations_df["feedback_score"]
    .mean()
)

print(
    f"Average Feedback Score: {avg_feedback_score:.4f}"
)

In [ ]:
# The positive feedback rate measures the percentage of recommendations that received a Like response.

positive_rate = (
    (recommendations_df["feedback"] == "Like")
    .mean()
    * 100
)

print(
    f"Positive Feedback Rate: {positive_rate:.2f}%"
)


In [ ]:
# Neutral feedback indicates that recommendations neither strongly matched nor strongly missed user preferences.

neutral_rate = (
    (recommendations_df["feedback"] == "Neutral")
    .mean()
    * 100
)

print(
    f"Neutral Feedback Rate: {neutral_rate:.2f}%"
)

In [ ]:
negative_rate = (
    (recommendations_df["feedback"] == "Dislike")
    .mean()
    * 100
)

print(
    f"Negative Feedback Rate: {negative_rate:.2f}%"
)

Feedback Evaluation Summary

The key feedback metrics are summarized in a single table for easier interpretation.

In [ ]:
feedback_summary = pd.DataFrame({
    "Metric": [
        "Average Feedback Score",
        "Positive Feedback Rate (%)",
        "Neutral Feedback Rate (%)",
        "Negative Feedback Rate (%)"
    ],
    "Value": [
        round(avg_feedback_score, 4),
        round(positive_rate, 2),
        round(neutral_rate, 2),
        round(negative_rate, 2)
    ]
})

feedback_summary

In [ ]:
'''The Feedback System assigns an updated rank to every recommendation after incorporating user feedback.

Analyzing the rank distribution helps verify that recommendations have been ordered correctly for each user.'''

recommendations_df["updated_rank"].describe()

In [ ]:
'''The Personalization Engine generates the final recommendation order using the personalization score.

This rank represents the final order shown to users.'''

recommendations_df["personalized_rank"].describe()

The recommendation pipeline contains two ranking stages:

- Updated Rank (after feedback)
- Personalized Rank (after personalization)

Comparing these rankings helps verify the transition from feedback-adjusted recommendations to the final personalized recommendation list.

In [ ]:
recommendations_df[
    [
        "visitorid",
        "itemid",
        "updated_rank",
        "personalized_rank",
        "updated_score",
        "personalization_score"
    ]
].head(20)

In [ ]:
# The personalization score is the final confidence score used for ranking recommendations.

plt.figure(figsize=(8,5))

plt.hist(
    recommendations_df["personalization_score"],
    bins=20
)

plt.title("Distribution of Personalization Scores")

plt.xlabel("Personalization Score")

plt.ylabel("Frequency")

plt.show()

In [ ]:
top_recommendations = (
    recommendations_df
    .sort_values(
        "personalization_score",
        ascending=False
    )
)

top_recommendations.head(20)

In [ ]:
recommendations_df[
    recommendations_df["personalized_rank"] <= 10
].head(20)

In [ ]:
rank_scores = (
    recommendations_df
    .groupby("personalized_rank")["personalization_score"]
    .mean()
)

rank_scores.head(10)

In [ ]:
''' Ranking Quality Visualization

The relationship between personalized rank and personalization score is visualized using a line chart.'''

plt.figure(figsize=(9,5))

plt.plot(
    rank_scores.index,
    rank_scores.values,
    marker="o"
)

plt.title("Average Personalization Score by Rank")

plt.xlabel("Personalized Rank")

plt.ylabel("Average Personalization Score")

plt.grid(True)

plt.show()

## Ranking Evaluation Summary

The ranking evaluation confirms that:

- Recommendations are ordered using personalization scores.
- Higher-ranked products generally receive higher personalization scores.
- Each user receives an ordered recommendation list.
- The final personalized ranking is ready for deployment through the recommendation API.